# Train Pace & Degradation Models

**Objective:** Train ML models to predict `LapTime` based on tyre age, fuel load, and compound.
**Input:** `backend/data/raw/Bahrain_2021_2023_laps.parquet`
**Output:** `backend/app/models/pace_model_v1.pkl`

In [ ]:
import pandas as pd
import numpy as np
from sklearn.linear_model import Ridge
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score
import joblib
import sys
import os

# Add backend to path
sys.path.append(os.path.abspath('../backend'))
from app.ml.features import prepare_features

## 1. Load Data

In [ ]:
DATA_PATH = '../backend/data/raw/Bahrain_2021_2023_laps.parquet'
df = pd.read_parquet(DATA_PATH)
print(f"Loaded {len(df)} laps.")

## 2. Feature Engineering

In [ ]:
df_clean = prepare_features(df)
print("Cleaned Laps:", len(df_clean))
print(df_clean[['Driver', 'LapTime', 'TyreAge', 'FuelPenalty', 'CompoundIndex']].head())

## 3. Train Base Pace Model
We use Ridge Regression to avoid overfitting.

In [ ]:
features = ['CompoundIndex', 'TyreAge', 'FuelPenalty']
target = 'LapTime' # We might need to convert Timedelta to float seconds first if fetch script didn't

# Ensure LapTime is float
if df_clean[target].dtype == 'timedelta64[ns]':
    df_clean[target] = df_clean[target].dt.total_seconds()

X = df_clean[features]
y = df_clean[target]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

model = Ridge(alpha=1.0)
model.fit(X_train, y_train)

# Evaluate
y_pred = model.predict(X_test)
print(f"R2 Score: {r2_score(y_test, y_pred):.3f}")
print(f"RMSE: {np.sqrt(mean_squared_error(y_test, y_pred)):.3f} seconds")

## 4. Save Model

In [ ]:
MODEL_DIR = '../backend/app/models'
os.makedirs(MODEL_DIR, exist_ok=True)
joblib.dump(model, os.path.join(MODEL_DIR, 'pace_model_v1.pkl'))
print("Model saved.")